# LeetCode 1048 — Longest String Chain bằng Python

Link đề: https://leetcode.com/problems/longest-string-chain/ · Độ khó: **Medium** · Chủ đề: Array, Hash Table, Two Pointers, String, Dynamic Programming, Sorting

Skeleton LeetCode:

```python
class Solution:
    def longestStrChain(self, words: List[str]) -> int:
        ...
```

Ghi chú: đáp án là **một số nguyên duy nhất** (độ dài chuỗi từ dài nhất). Điểm cốt lõi: `a` là tiền bối của `b` ⇔ `len(a) + 1 = len(b)` và xoá được **đúng một** ký tự của `b` để thành `a`. Quan hệ này chỉ đi từ từ ngắn sang từ dài hơn đúng `1` nên đồ thị là **DAG phân tầng theo độ dài** ⇒ sắp xếp theo độ dài rồi **DP**: `dp[w] = 1 + max dp[w xoá một ký tự]`.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: tiền bối (predecessor), chuỗi từ (word chain), độ dài chuỗi dài nhất chọn từ danh sách `words`.
2. Nắm công cụ: sắp xếp theo độ dài, DP trên DAG (đường đi dài nhất), bảng băm `dict` từ → giá trị DP, kiểm tra "xoá đúng một ký tự" bằng hai con trỏ hoặc bằng cắt chuỗi.
3. Đi từ **vét cạn DFS mọi chuỗi** → **DP so sánh từng cặp (hai con trỏ)** → **DP băm: xoá từng ký tự** → **đệ quy có nhớ** → **DP đẩy: chèn từng chữ cái**.
4. Hiểu mấu chốt: chuỗi dài hơn chỉ nhận tiền bối từ tầng độ dài ngay dưới; thay vì thử mọi cặp (`n²`) ta sinh **tối đa `L` tiền bối ứng viên** của mỗi từ (xoá từng vị trí) và tra bảng băm ⇒ `O(n · L²)`.
5. Tránh lỗi kinh điển: không sắp xếp theo độ dài, dùng "dãy con" mà quên điều kiện độ dài hơn kém `1`, chỉ xoá đầu/cuối, đếm số độ dài khác nhau thay vì chuỗi thật, quên cộng `1` cho chính từ đó, `KeyError` khi tiền bối không có trong từ điển.

## 2. Đề bài & ví dụ

Cho mảng `words` gồm các từ chỉ chứa chữ cái thường.

`wordA` là **tiền bối** của `wordB` nếu và chỉ nếu ta có thể **chèn đúng một chữ cái vào bất kỳ vị trí nào** của `wordA`, không đổi thứ tự các ký tự còn lại, để được `wordB`. Ví dụ `"abc"` là tiền bối của `"abac"`, còn `"cba"` **không** là tiền bối của `"bcad"`.

Một **chuỗi từ** là dãy `[word_1, …, word_k]` (`k ≥ 1`) trong đó `word_1` là tiền bối của `word_2`, `word_2` là tiền bối của `word_3`, … Một từ đơn lẻ là chuỗi trivial `k = 1`.

Trả về **độ dài** của chuỗi từ **dài nhất** có thể tạo từ các từ trong `words`.

**Ví dụ 1:** `words = ["a","b","ba","bca","bda","bdca"]` → `4` (chuỗi `["a","ba","bda","bdca"]`).

**Ví dụ 2:** `words = ["xbc","pcxbcf","xb","cxbc","pcxbc"]` → `5` (`["xb","xbc","cxbc","pcxbc","pcxbcf"]`).

**Ví dụ 3:** `words = ["abcd","dbqca"]` → `1` (`["abcd","dbqca"]` không hợp lệ vì đổi thứ tự chữ).

**Ràng buộc:** `1 <= words.length <= 1000`; `1 <= words[i].length <= 16`; `words[i]` chỉ gồm chữ cái thường.

In [1]:
import itertools
import random
import time
from functools import lru_cache
from string import ascii_lowercase
from typing import List

examples = [
    (["a", "b", "ba", "bca", "bda", "bdca"], 4),
    (["xbc", "pcxbcf", "xb", "cxbc", "pcxbc"], 5),
    (["abcd", "dbqca"], 1),
]
for words, want in examples:
    print(f"words = {words} -> {want}")

words = ['a', 'b', 'ba', 'bca', 'bda', 'bdca'] -> 4
words = ['xbc', 'pcxbcf', 'xb', 'cxbc', 'pcxbc'] -> 5
words = ['abcd', 'dbqca'] -> 1


## 3. Kiến thức nền

### 3.1 Ba cách phát biểu "tiền bối" (tương đương)

Với hai từ `a`, `b`, ba điều sau **tương đương**:

1. **Định nghĩa chèn:** tồn tại vị trí `i` và chữ cái `c` sao cho `a[:i] + c + a[i:] == b`.
2. **Xoá một ký tự:** `len(a) + 1 == len(b)` và tồn tại `i` sao cho `b[:i] + b[i+1:] == a`.
3. **Dãy con + độ dài:** `len(a) + 1 == len(b)` và `a` là **dãy con** (subsequence) của `b`.

Lý do: (1) ⇔ (2) là hai chiều của cùng một phép biến đổi. (2) ⇒ (3): xoá một ký tự của `b` giữ nguyên thứ tự nên `a` là dãy con. (3) ⇒ (2): `a` là dãy con của `b` mà kém `b` đúng `1` ký tự nên có đúng một ký tự của `b` không được khớp; xoá nó ra được `a`. Ta kiểm bằng máy trên mọi cặp từ nhỏ ở mục 12.

In [2]:
@lru_cache(maxsize=None)
def pred_by_insert(a, b):
    """Định nghĩa gốc: chèn một chữ cái bất kỳ vào một vị trí bất kỳ của a để được b."""
    if len(a) + 1 != len(b):
        return False
    return any(a[:i] + c + a[i:] == b for i in range(len(a) + 1) for c in ascii_lowercase)


def pred_by_delete(a, b):
    """Xoá đúng một ký tự của b để được a."""
    return len(a) + 1 == len(b) and any(b[:i] + b[i + 1:] == a for i in range(len(b)))


def pred_two_pointers(a, b):
    """a là dãy con của b và hơn kém đúng 1 ký tự (hai con trỏ)."""
    if len(a) + 1 != len(b):
        return False
    i = 0
    for c in b:
        if i < len(a) and a[i] == c:
            i += 1
    return i == len(a)


for a, b, want in [("abc", "abac", True), ("cba", "bcad", False), ("a", "ba", True), ("ba", "bca", True), ("ab", "ab", False), ("a", "abc", False)]:
    assert pred_by_insert(a, b) == pred_by_delete(a, b) == pred_two_pointers(a, b) == want, (a, b)
print("3 định nghĩa tương đương trên các cặp mẫu")

3 định nghĩa tương đương trên các cặp mẫu


### 3.2 DAG phân tầng theo độ dài

Cạnh `a → b` (a là tiền bối của b) luôn tăng độ dài đúng `1` nên **không có chu trình**. Chuỗi từ dài nhất là **đường đi dài nhất trong DAG** (tính theo số đỉnh). Nếu xử lý các từ theo độ dài tăng dần thì khi tới `b`, mọi tiền bối của `b` (độ dài `len(b) - 1`) đã được xử lý.

### 3.3 Từ trùng nhau

Đề không cấm `words` chứa từ trùng. Hai từ trùng **không** thể nối nhau (độ dài không hơn kém `1`) nên từ trùng không làm đáp án tăng; ta cứ để thuật toán xử lý bình thường (bảng băm ghi đè cùng giá trị).

## 4. Ý tưởng nền tảng / trực giác

Hình dung ta xây chuỗi từ **dưới lên**: bắt đầu từ từ ngắn, mỗi bước thêm một chữ cái để được từ dài hơn **có trong danh sách**. Với một từ `w`, chuỗi dài nhất **kết thúc tại `w`** phải đi qua một tiền bối `p` của `w` (xoá `1` ký tự của `w`) rồi nối thêm `w`: độ dài `= 1 +` (chuỗi dài nhất kết thúc tại `p`). Mỗi từ dài `L` chỉ có tối đa `L` ứng viên tiền bối (xoá từng vị trí), nên chỉ cần tra tối đa `L` lần bảng băm.

## 5. Brute force — DFS mọi chuỗi

**Ý tưởng:** bắt đầu từ từng từ, đệ quy thử mọi từ tiếp theo là "hậu duệ" (từ mà từ hiện tại là tiền bối của nó) theo **định nghĩa chèn**, ghi nhận độ dài dài nhất. Không dùng nhớ, không dùng sắp xếp: đúng theo định nghĩa nên làm **oracle** cho danh sách nhỏ.

In [3]:
def chain_brute(words):
    best = 1

    def dfs(cur, length):
        nonlocal best
        best = max(best, length)
        for nxt in words:
            if pred_by_insert(cur, nxt):
                dfs(nxt, length + 1)

    for w in words:
        dfs(w, 1)
    return best


for words, want in examples:
    assert chain_brute(words) == want
print([chain_brute(w) for w, _ in examples])

[4, 5, 1]


In [4]:
# Code nộp LeetCode cho Brute force (đúng nhưng chậm: có thể thử lại cùng một chuỗi con rất nhiều lần)
class SolutionBrute:
    def longestStrChain(self, words: List[str]) -> int:
        def pred(a, b):
            if len(a) + 1 != len(b):
                return False
            return any(a[:i] + c + a[i:] == b for i in range(len(a) + 1) for c in "abcdefghijklmnopqrstuvwxyz")

        best = 1

        def dfs(cur, length):
            nonlocal best
            best = max(best, length)
            for nxt in words:
                if pred(cur, nxt):
                    dfs(nxt, length + 1)

        for w in words:
            dfs(w, 1)
        return best

**Độ phức tạp:** số đường đi trong DAG có thể rất lớn (mũ); mỗi bước kiểm tra tiền bối tốn `O(26 · L²)`.

**Nút thắt:** cùng một từ `cur` được mở rộng lại nhiều lần từ các đường khác nhau — chuỗi dài nhất bắt đầu từ `cur` luôn như nhau.

## 6. Bước đột phá

Gọi `dp[w]` = độ dài chuỗi dài nhất **kết thúc tại** `w`. Mọi tiền bối của `w` có độ dài nhỏ hơn đúng `1`, nên nếu xử lý theo độ dài tăng dần thì `dp[w] = 1 + max(dp[p])` trên các `p` là tiền bối của `w` có trong danh sách. Tiền bối của `w` chính là các từ nhận được bằng cách xoá một ký tự — chỉ `≤ L` ứng viên, tra bảng băm `O(L)` mỗi ứng viên.

## 7. Các cách tiếp theo

Ngoài Brute force (mục 5), ta có thêm 4 cách: **Cách 2** DP sắp xếp + so sánh từng cặp bằng hai con trỏ, **Cách 3** DP bảng băm + xoá từng ký tự (đáp án chính), **Cách 4** đệ quy có nhớ, **Cách 5** DP đẩy bằng cách chèn từng chữ cái.

### Cách 2 — DP sắp xếp theo độ dài + hai con trỏ

**Giải thích:** sắp xếp `words` theo độ dài tăng dần; `f[i]` = chuỗi dài nhất kết thúc tại `words[i]`, khởi tạo `1`. Với mỗi `i`, duyệt mọi `j < i`; nếu `words[j]` là tiền bối của `words[i]` (kiểm tra hai con trỏ) thì `f[i] = max(f[i], f[j] + 1)`. Đáp án `max(f)`.

In [5]:
def chain_two_pointers(words):
    ws = sorted(words, key=len)
    n = len(ws)
    f = [1] * n
    for i in range(n):
        for j in range(i):
            if pred_two_pointers(ws[j], ws[i]):
                f[i] = max(f[i], f[j] + 1)
    return max(f)


for words, want in examples:
    assert chain_two_pointers(words) == want
print([chain_two_pointers(w) for w, _ in examples])

[4, 5, 1]


In [6]:
# Code nộp LeetCode cho Cách 2
class SolutionTwoPointers:
    def longestStrChain(self, words: List[str]) -> int:
        def check(a: str, b: str) -> bool:
            if len(a) + 1 != len(b):
                return False
            i = 0
            for c in b:
                if i < len(a) and a[i] == c:
                    i += 1
            return i == len(a)

        words = sorted(words, key=len)
        n = len(words)
        f = [1] * n
        for i in range(n):
            for j in range(i):
                if check(words[j], words[i]):
                    f[i] = max(f[i], f[j] + 1)
        return max(f)

**Độ phức tạp:** `O(n log n + n² · L)` thời gian, `O(n)` bộ nhớ (`n ≤ 1000`, `L ≤ 16` nên khoảng `10^7` thao tác).

### Cách 3 — DP bảng băm, xoá từng ký tự (đáp án chính)

**Ý tưởng:** sắp xếp theo độ dài; `dp` là `dict` từ → độ dài chuỗi dài nhất kết thúc tại từ đó. Với mỗi từ `w` (theo độ dài tăng), với mỗi vị trí `i`, tiền bối ứng viên `p = w[:i] + w[i+1:]`; nếu `p` đã có trong `dp` thì `dp[w] = max(dp[w], dp[p] + 1)`.

In [7]:
def chain_hash(words):
    dp = {}
    best = 1
    for w in sorted(words, key=len):
        cur = 1
        for i in range(len(w)):
            p = w[:i] + w[i + 1:]                          # xoá ký tự thứ i
            if p in dp:                                    # tiền bối có trong danh sách (độ dài ngắn hơn 1: đã xử lý)
                cur = max(cur, dp[p] + 1)
        dp[w] = cur
        best = max(best, cur)
    return best


for words, want in examples:
    assert chain_hash(words) == want
print([chain_hash(w) for w, _ in examples])

[4, 5, 1]


In [8]:
# Code nộp LeetCode cho Cách 3
class SolutionHash:
    def longestStrChain(self, words: List[str]) -> int:
        dp = {}
        best = 1
        for w in sorted(words, key=len):
            cur = 1
            for i in range(len(w)):
                p = w[:i] + w[i + 1:]
                if p in dp:
                    cur = max(cur, dp[p] + 1)
            dp[w] = cur
            best = max(best, cur)
        return best

**Độ phức tạp:** `O(n log n + n · L²)` thời gian (mỗi từ có `L` ứng viên, mỗi lần cắt và băm chuỗi tốn `O(L)`), `O(n · L)` bộ nhớ cho từ điển.

### Cách 4 — Đệ quy có nhớ (top-down)

**Ý tưởng:** `go(w)` = chuỗi dài nhất kết thúc tại `w`: `1 + max go(p)` trên các tiền bối `p ∈ S` (xoá một ký tự). Không cần sắp xếp; đệ quy sâu tối đa `L = 16` tầng. Đáp án `max go(w)`.

In [9]:
def chain_topdown(words):
    S = set(words)

    @lru_cache(maxsize=None)
    def go(w):
        best = 1
        for i in range(len(w)):
            p = w[:i] + w[i + 1:]
            if p in S:
                best = max(best, go(p) + 1)
        return best

    return max(go(w) for w in S)


for words, want in examples:
    assert chain_topdown(words) == want
print([chain_topdown(w) for w, _ in examples])

[4, 5, 1]


In [10]:
# Code nộp LeetCode cho Cách 4
class SolutionTopDown:
    def longestStrChain(self, words: List[str]) -> int:
        S = set(words)

        @lru_cache(maxsize=None)
        def go(w):
            best = 1
            for i in range(len(w)):
                p = w[:i] + w[i + 1:]
                if p in S:
                    best = max(best, go(p) + 1)
            return best

        return max(go(w) for w in S)

**Độ phức tạp:** `O(n · L²)` thời gian, `O(n · L)` bộ nhớ (bảng nhớ + tập từ + ngăn xếp `≤ 16`).

### Cách 5 — DP đẩy: chèn từng chữ cái

**Ý tưởng:** đi từ tiền bối **sang hậu duệ**. `dp` khởi tạo `1` cho mọi từ; duyệt từ theo độ dài tăng; với mỗi từ `w`, sinh mọi từ dài hơn bằng cách **chèn một chữ cái `a..z` vào một trong `L+1` vị trí**; nếu từ đó có trong danh sách thì `dp[nx] = max(dp[nx], dp[w] + 1)`. Khi tới `w`, mọi tiền bối (ngắn hơn) đã đẩy xong nên `dp[w]` là chốt.

In [11]:
def chain_push(words):
    dp = {w: 1 for w in words}
    for w in sorted(dp, key=len):
        for i in range(len(w) + 1):
            for c in ascii_lowercase:
                nx = w[:i] + c + w[i:]
                if nx in dp and dp[nx] < dp[w] + 1:
                    dp[nx] = dp[w] + 1
    return max(dp.values())


for words, want in examples:
    assert chain_push(words) == want
print([chain_push(w) for w, _ in examples])

[4, 5, 1]


In [12]:
# Code nộp LeetCode cho Cách 5
class SolutionPush:
    def longestStrChain(self, words: List[str]) -> int:
        dp = {w: 1 for w in words}
        for w in sorted(dp, key=len):
            for i in range(len(w) + 1):
                for c in "abcdefghijklmnopqrstuvwxyz":
                    nx = w[:i] + c + w[i:]
                    if nx in dp and dp[nx] < dp[w] + 1:
                        dp[nx] = dp[w] + 1
        return max(dp.values())

**Độ phức tạp:** `O(n log n + 26 · n · L²)` thời gian, `O(n · L)` bộ nhớ. Hằng số lớn hơn Cách 3 (`26` lần) nhưng không cần so sánh cặp.

## 8. TRACE từng bước

Trace **Cách 3** với `words = ["a","b","ba","bca","bda","bdca"]` (đã sắp theo độ dài: `a, b, ba, bca, bda, bdca`):

| từ `w` | tiền bối ứng viên (xoá từng vị trí) | có trong `dp`? | `dp[w]` |
|---|---|---|---|
| `a` | `""` | không | 1 |
| `b` | `""` | không | 1 |
| `ba` | `a`, `b` | `a`:1, `b`:1 | 2 |
| `bca` | `ca`, `ba`, `bc` | `ba`:2 | 3 |
| `bda` | `da`, `ba`, `bd` | `ba`:2 | 3 |
| `bdca` | `dca`, `bca`, `bda`, `bdc` | `bca`:3, `bda`:3 | 4 |

Đáp án `max = 4`; một chuỗi: `a → ba → bda → bdca`. Hàm trace in từng bước:

In [13]:
def trace_hash(words):
    dp = {}
    print(f"words = {words}")
    for w in sorted(words, key=len):
        cands = [w[:i] + w[i + 1:] for i in range(len(w))]
        hits = {p: dp[p] for p in cands if p in dp}
        dp[w] = 1 + max(hits.values(), default=0)
        print(f"  {w!r:9}: ứng viên {cands} -> có trong dp {hits or '(không)'} -> dp = {dp[w]}")
    best = max(dp.values())
    print("  đáp án =", best)
    return best


assert trace_hash(["a", "b", "ba", "bca", "bda", "bdca"]) == 4
assert trace_hash(["abcd", "dbqca"]) == 1

words = ['a', 'b', 'ba', 'bca', 'bda', 'bdca']
  'a'      : ứng viên [''] -> có trong dp (không) -> dp = 1
  'b'      : ứng viên [''] -> có trong dp (không) -> dp = 1
  'ba'     : ứng viên ['a', 'b'] -> có trong dp {'a': 1, 'b': 1} -> dp = 2
  'bca'    : ứng viên ['ca', 'ba', 'bc'] -> có trong dp {'ba': 2} -> dp = 3
  'bda'    : ứng viên ['da', 'ba', 'bd'] -> có trong dp {'ba': 2} -> dp = 3
  'bdca'   : ứng viên ['dca', 'bca', 'bda', 'bdc'] -> có trong dp {'bca': 3, 'bda': 3} -> dp = 4
  đáp án = 4
words = ['abcd', 'dbqca']
  'abcd'   : ứng viên ['bcd', 'acd', 'abd', 'abc'] -> có trong dp (không) -> dp = 1
  'dbqca'  : ứng viên ['bqca', 'dqca', 'dbca', 'dbqa', 'dbqc'] -> có trong dp (không) -> dp = 1
  đáp án = 1


**Trace ví dụ 2** (`["xbc","pcxbcf","xb","cxbc","pcxbc"]`): sắp theo độ dài `xb, xbc, cxbc, pcxbc, pcxbcf`: `dp[xb] = 1`, `dp[xbc] = dp[xb] + 1 = 2`, `dp[cxbc] = dp[xbc] + 1 = 3`, `dp[pcxbc] = dp[cxbc] + 1 = 4`, `dp[pcxbcf] = dp[pcxbc] + 1 = 5`.

In [14]:
assert trace_hash(["xbc", "pcxbcf", "xb", "cxbc", "pcxbc"]) == 5

words = ['xbc', 'pcxbcf', 'xb', 'cxbc', 'pcxbc']
  'xb'     : ứng viên ['b', 'x'] -> có trong dp (không) -> dp = 1
  'xbc'    : ứng viên ['bc', 'xc', 'xb'] -> có trong dp {'xb': 1} -> dp = 2
  'cxbc'   : ứng viên ['xbc', 'cbc', 'cxc', 'cxb'] -> có trong dp {'xbc': 2} -> dp = 3
  'pcxbc'  : ứng viên ['cxbc', 'pxbc', 'pcbc', 'pcxc', 'pcxb'] -> có trong dp {'cxbc': 3} -> dp = 4
  'pcxbcf' : ứng viên ['cxbcf', 'pxbcf', 'pcbcf', 'pcxcf', 'pcxbf', 'pcxbc'] -> có trong dp {'pcxbc': 4} -> dp = 5
  đáp án = 5


**Khôi phục chuỗi:** lưu `parent[w]` = tiền bối đạt max rồi lần ngược từ từ có `dp` lớn nhất.

In [15]:
def best_chain(words):
    dp, parent = {}, {}
    for w in sorted(words, key=len):
        dp[w], parent[w] = 1, None
        for i in range(len(w)):
            p = w[:i] + w[i + 1:]
            if p in dp and dp[p] + 1 > dp[w]:
                dp[w], parent[w] = dp[p] + 1, p
    end = max(dp, key=dp.get)
    chain = []
    while end is not None:
        chain.append(end)
        end = parent[end]
    return chain[::-1]


def valid_chain(chain, words):
    """Kiểm tra theo ĐỊNH NGHĨA: mọi từ thuộc words, mỗi từ là tiền bối của từ kế tiếp."""
    return all(w in words for w in chain) and all(pred_by_insert(chain[i], chain[i + 1]) for i in range(len(chain) - 1))


ch = best_chain(["a", "b", "ba", "bca", "bda", "bdca"])
print(ch)
assert len(ch) == 4 and valid_chain(ch, ["a", "b", "ba", "bca", "bda", "bdca"])

['a', 'ba', 'bca', 'bdca']


## 9. Code nộp LeetCode

Chọn **Cách 3** (DP bảng băm, xoá từng ký tự). Giữ nguyên `class Solution`, `longestStrChain(self, words)`:

In [16]:
class Solution:
    def longestStrChain(self, words: List[str]) -> int:
        dp = {}                                         # dp[w] = chuỗi dài nhất kết thúc tại w
        best = 1
        for w in sorted(words, key=len):                # ngắn trước: mọi tiền bối của w đã được xử lý
            cur = 1                                     # chính w luôn là chuỗi độ dài 1
            for i in range(len(w)):
                p = w[:i] + w[i + 1:]                   # xoá ký tự thứ i => tiền bối ứng viên
                if p in dp:
                    cur = max(cur, dp[p] + 1)
            dp[w] = cur
            best = max(best, cur)
        return best


print([Solution().longestStrChain(list(w)) for w, _ in examples])
assert all(Solution().longestStrChain(list(w)) == v for w, v in examples)

[4, 5, 1]


## 10. Vì sao đúng

**(a) Tương đương định nghĩa.** Như mục 3.1: `a` là tiền bối của `b` ⇔ `len(a) + 1 = len(b)` và `a = b` bỏ đi đúng một ký tự. Vậy tập tiền bối của `w` có trong danh sách chính là `{w[:i] + w[i+1:]} ∩ words`.

**(b) DAG và thứ tự xử lý.** Mọi tiền bối của `w` có độ dài `len(w) - 1`. Sắp xếp theo độ dài tăng dần (ổn định) bảo đảm khi xử lý `w`, mọi từ ngắn hơn `w` đã có mặt trong `dp`. Các từ **cùng độ dài** với `w` không bao giờ là tiền bối của `w`, nên thứ tự giữa chúng không ảnh hưởng.

**(c) Phương trình DP.** Chuỗi dài nhất kết thúc tại `w`: hoặc chỉ gồm `w` (độ dài `1`), hoặc phần trước `w` là một chuỗi kết thúc tại tiền bối `p` của `w`. Chuỗi kết thúc tại `p` tối ưu cho độ dài `dp[p]`; ghép với `w` hợp lệ vì `p` là tiền bối của `w`. Vậy `dp[w] = 1 + max(0, max_p dp[p])`. Bằng quy nạp theo độ dài từ, `dp[w]` đúng. Đáp án là `max_w dp[w]` vì chuỗi dài nhất phải kết thúc ở một từ nào đó.

**(d) Từ trùng.** Bảng băm ghi `dp[w]` nhiều lần với cùng giá trị (các tiền bối giống nhau) nên không đổi kết quả. Hai từ trùng không nối được với nhau (cần hơn kém `1` độ dài).

**(e) Các cách còn lại.** Cách 2 dùng định nghĩa (3) trong mục 3.1 và duyệt mọi cặp; Cách 4 là bản đệ quy của (c); Cách 5 là bản "đẩy" tuân theo đúng thứ tự (b): khi tới `w`, tất cả tiền bối đã đẩy xong. Cách 1 là định nghĩa.

**(f) Kiểm chứng bằng máy.** Mọi cách khớp oracle DFS-định-nghĩa trên mọi tập con của `14` từ ngắn (`16 383` tập khác rỗng), `12` từ (`4 095` tập khác rỗng), trên `3 000` danh sách ngẫu nhiên nhỏ và khớp nhau trên `n = 1000`; chuỗi khôi phục luôn hợp lệ theo định nghĩa.

## 11. Time / Space Complexity

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. DFS vét cạn | mũ (số đường đi) | `O(n)` | oracle `n` nhỏ |
| 2. Hai con trỏ | `O(n log n + n² L)` | `O(n)` | đơn giản, `n ≤ 1000` đủ nhanh |
| 3. Băm + xoá ký tự | `O(n log n + n L²)` | `O(n L)` | **đáp án chính** |
| 4. Đệ quy có nhớ | `O(n L²)` | `O(n L)` | không cần sắp xếp |
| 5. Đẩy: chèn chữ cái | `O(26 n L²)` | `O(n L)` | cách nhìn "hậu duệ" |

Đo thực tế với `n = 1000` (các từ nhị phân độ dài `≤ 9` — nhiều quan hệ tiền bối — và từ ngẫu nhiên độ dài `≤ 16`):

In [17]:
def bench(fn, arg, repeat=2):
    best = 1e9
    for _ in range(repeat):
        t0 = time.perf_counter()
        fn(list(arg))
        best = min(best, time.perf_counter() - t0)
    return best


binary = ["".join(t) for L in range(1, 10) for t in itertools.product("ab", repeat=L)][:1000]
rng = random.Random(1048)
rand = ["".join(rng.choice("abcdefgh") for _ in range(rng.randint(1, 16))) for _ in range(1000)]
for label, ws in [("nhị phân", binary), ("ngẫu nhiên", rand)]:
    print(label, "n =", len(ws))
    for name, fn in [("two_pointers", chain_two_pointers), ("hash", chain_hash), ("topdown", chain_topdown), ("push", chain_push)]:
        print(f"  {name:13s}: {bench(fn, ws) * 1000:8.1f} ms -> {fn(list(ws))}")

nhị phân n = 1000


  two_pointers :    126.9 ms -> 9
  hash         :      1.7 ms -> 9
  topdown      :      2.1 ms -> 9
  push         :     35.6 ms -> 9
ngẫu nhiên n = 1000


  two_pointers :     70.4 ms -> 4
  hash         :      1.5 ms -> 4
  topdown      :      1.6 ms -> 4
  push         :     36.3 ms -> 4


## 12. Bộ test

Chia 6 nhóm: (A) ví dụ đề, (B) edge case, (C) phản ví dụ cho lỗi hay gặp, (D) vét cạn mọi tập từ nhỏ, (E) tính chất / ngẫu nhiên (seed cố định), (F) giới hạn.

### A. Ví dụ của đề

In [18]:
IMPLS = {
    "two_pointers": chain_two_pointers,
    "hash": chain_hash,
    "topdown": chain_topdown,
    "push": chain_push,
    "Solution": lambda w: Solution().longestStrChain(w),
    "SolutionTwoPointers": lambda w: SolutionTwoPointers().longestStrChain(w),
    "SolutionHash": lambda w: SolutionHash().longestStrChain(w),
    "SolutionTopDown": lambda w: SolutionTopDown().longestStrChain(w),
    "SolutionPush": lambda w: SolutionPush().longestStrChain(w),
}
BRUTE = {"brute": chain_brute, "SolutionBrute": lambda w: SolutionBrute().longestStrChain(w)}


def check(words, want=None, brute=False):
    if want is None:
        want = chain_brute(list(words)) if brute else Solution().longestStrChain(list(words))
    impls = dict(IMPLS)
    if brute:
        impls.update(BRUTE)
    for name, fn in impls.items():
        got = fn(list(words))
        assert got == want, (name, words, got, want)
    return want


for words, want in examples:
    check(words, want, brute=True)
print("ví dụ đề: mọi cài đặt đúng;", len(IMPLS), "DP +", len(BRUTE), "vét cạn")

ví dụ đề: mọi cài đặt đúng; 9 DP + 2 vét cạn


### B. Edge case

- Một từ duy nhất (`["a"]`, từ dài `16` ký tự).
- Mọi từ cùng độ dài ⇒ không có chuỗi nào dài hơn `1`.
- Từ trùng nhau (`["a","a"]`, `["a","a","ab"]`).
- Chuỗi nằm rải rác, thứ tự đầu vào đảo ngược (dài trước).
- Hai từ có cùng ký tự nhưng đổi thứ tự (`abcd` / `dbqca`).
- Một tiền bối có nhiều hậu duệ; một hậu duệ có nhiều tiền bối (hợp chuỗi).
- Chữ cái lặp (`"a"`, `"aa"`, `"aaa"`): xoá ký tự nào cũng cho cùng tiền bối.
- Chuỗi dài `16` (`a`, `ab`, `abc`, … ).
- Test LeetCode nổi tiếng (`ksqvsyq`, …).

In [19]:
edge = [
    (["a"], 1), (["z" * 16], 1), (["ab", "cd", "ef"], 1), (["a", "a"], 1), (["a", "a", "ab"], 2),
    (["abc", "ab", "a"], 3), (["abcd", "dbqca"], 1), (["a", "aa", "aaa", "aaaa"], 4),
    (["a", "ab", "ac", "abc"], 3), (["a", "b", "ab", "ba", "aba"], 3), (["bca", "ba", "a"], 3),
    (["a", "bb"], 1), (["ab", "b"], 2), (["abc", "bc", "c"], 3),
    (["".join(chr(97 + j) for j in range(i)) for i in range(1, 17)], 16),
    (["".join(chr(97 + j) for j in range(i)) for i in range(16, 0, -1)], 16),
    (["ksqvsyq", "ks", "kss", "czvh", "zczpzvdhx", "zczpzvh", "zczpzvhx", "zcpzvh", "zczvh", "gr", "grukmj", "ksqvsq", "gruj", "kssq", "ksqsq", "grukkmj", "grukj", "zczpzfvdhx", "gru"], None),
]
for words, want in edge:
    got = check(words, want, brute=len(words) <= 20)
    print(f"  {len(words):2d} từ {str(words)[:50]:50s}... -> {got}")
print(len(edge), "edge case: mọi cài đặt (kể cả vét cạn) khớp")

   1 từ ['a']                                             ... -> 1
   1 từ ['zzzzzzzzzzzzzzzz']                              ... -> 1
   3 từ ['ab', 'cd', 'ef']                                ... -> 1
   2 từ ['a', 'a']                                        ... -> 1
   3 từ ['a', 'a', 'ab']                                  ... -> 2
   3 từ ['abc', 'ab', 'a']                                ... -> 3
   2 từ ['abcd', 'dbqca']                                 ... -> 1
   4 từ ['a', 'aa', 'aaa', 'aaaa']                        ... -> 4
   4 từ ['a', 'ab', 'ac', 'abc']                          ... -> 3
   5 từ ['a', 'b', 'ab', 'ba', 'aba']                     ... -> 3
   3 từ ['bca', 'ba', 'a']                                ... -> 3
   2 từ ['a', 'bb']                                       ... -> 1
   2 từ ['ab', 'b']                                       ... -> 2
   3 từ ['abc', 'bc', 'c']                                ... -> 3
  16 từ ['a', 'ab', 'abc', 'abcd', 'abcde', 'abcdef', 'abc... 

### C. Phản ví dụ: cài đặt sai phải bị bắt

In [20]:
def bug_no_sort(words):                                  # DP theo thứ tự đầu vào, không sắp xếp theo độ dài
    n = len(words)
    f = [1] * n
    for i in range(n):
        for j in range(i):
            if pred_two_pointers(words[j], words[i]):
                f[i] = max(f[i], f[j] + 1)
    return max(f)


def bug_subsequence_only(words):                         # dãy con, bỏ điều kiện độ dài hơn kém đúng 1
    def sub(a, b):
        it = iter(b)
        return all(c in it for c in a)
    ws = sorted(words, key=len)
    f = [1] * len(ws)
    for i in range(len(ws)):
        for j in range(i):
            if sub(ws[j], ws[i]):
                f[i] = max(f[i], f[j] + 1)
    return max(f)


def bug_ends_only(words):                                # chỉ xoá ký tự đầu hoặc cuối
    dp = {}
    best = 1
    for w in sorted(words, key=len):
        cur = 1
        for p in (w[1:], w[:-1]):
            if p in dp:
                cur = max(cur, dp[p] + 1)
        dp[w] = cur
        best = max(best, cur)
    return best


def bug_distinct_lengths(words):                         # đếm số độ dài khác nhau
    return len(set(map(len, words)))


def bug_init_zero(words):                                # quên tính chính từ đó (khởi tạo 0)
    dp = {}
    best = 0
    for w in sorted(words, key=len):
        cur = 0
        for i in range(len(w)):
            p = w[:i] + w[i + 1:]
            if p in dp:
                cur = max(cur, dp[p] + 1)
        dp[w] = cur
        best = max(best, cur)
    return best


def bug_keyerror(words):                                 # dp[p] không kiểm tra p có trong từ điển
    dp = {}
    best = 1
    for w in sorted(words, key=len):
        cur = 1
        for i in range(len(w)):
            p = w[:i] + w[i + 1:]
            cur = max(cur, dp[p] + 1)
        dp[w] = cur
        best = max(best, cur)
    return best


BUGS = [bug_no_sort, bug_subsequence_only, bug_ends_only, bug_distinct_lengths, bug_init_zero, bug_keyerror]
rng = random.Random(10480)
pool = []
for _ in range(4000):
    n = rng.randint(1, 7)
    pool.append([("".join(rng.choice("abc") for _ in range(rng.randint(1, 4)))) for _ in range(n)])
for bug in BUGS:
    found = None
    for words in pool:
        try:
            got = bug(list(words))
        except KeyError:
            got = "KeyError"
        if got != chain_brute(words):
            found = (words, got)
            break
    assert found, bug.__name__
    print(f"{bug.__name__:22s} sai trên {found[0]}: ra {found[1]}, đúng {chain_brute(found[0])}")

bug_no_sort            sai trên ['babc', 'ac', 'ab', 'c', 'c', 'abbb']: ra 1, đúng 2
bug_subsequence_only   sai trên ['babc', 'ac', 'ab', 'c', 'c', 'abbb']: ra 4, đúng 2
bug_ends_only          sai trên ['cac', 'caac', 'cbca']: ra 1, đúng 2
bug_distinct_lengths   sai trên ['babc', 'ac', 'ab', 'c', 'c', 'abbb']: ra 3, đúng 2
bug_init_zero          sai trên ['bbbc']: ra 0, đúng 1
bug_keyerror           sai trên ['bbbc']: ra KeyError, đúng 1


### D. Vét cạn mọi tập từ nhỏ

Hai "vũ trụ" từ: (1) mọi chuỗi nhị phân `{a,b}` độ dài `1..3` (`2 + 4 + 8 = 14` từ ⇒ `16 384` tập con); (2) mọi chuỗi `{a,b,c}` độ dài `1..2` (`3 + 9 = 12` từ ⇒ `4 096` tập con). Với **mỗi** tập con (theo thứ tự đầu vào bị xáo), so mọi cài đặt DP với oracle DFS-định-nghĩa.

In [21]:
t0 = time.perf_counter()
cnt = 0
rng = random.Random(77)
for universe in (["".join(t) for L in range(1, 4) for t in itertools.product("ab", repeat=L)],
                 ["".join(t) for L in range(1, 3) for t in itertools.product("abc", repeat=L)]):
    m = len(universe)
    for mask in range(1, 1 << m):
        words = [universe[i] for i in range(m) if mask >> i & 1]
        rng.shuffle(words)
        want = chain_brute(words)
        for name, fn in IMPLS.items():
            assert fn(list(words)) == want, (name, words)
        cnt += 1
print(cnt, "tập từ:", len(IMPLS), "cài đặt DP khớp oracle DFS-định-nghĩa;", round(time.perf_counter() - t0, 1), "s")

20478 tập từ: 9 cài đặt DP khớp oracle DFS-định-nghĩa; 5.3 s


Kiểm tra thêm **ba định nghĩa tiền bối tương đương** (mục 3.1) trên **mọi cặp** từ của `{a,b,c}` độ dài `1..4`:

In [22]:
allw = ["".join(t) for L in range(1, 5) for t in itertools.product("abc", repeat=L)]
pairs = 0
for a in allw:
    for b in allw:
        r1, r2, r3 = pred_by_insert(a, b), pred_by_delete(a, b), pred_two_pointers(a, b)
        assert r1 == r2 == r3, (a, b)
        pairs += 1
print(len(allw), "từ,", pairs, "cặp: định nghĩa chèn = xoá một ký tự = dãy con kém 1 độ dài")

120 từ, 14400 cặp: định nghĩa chèn = xoá một ký tự = dãy con kém 1 độ dài


### E. Tính chất / ngẫu nhiên (seed cố định)

(1) Danh sách nhỏ (`n ≤ 11`): khớp oracle DFS; (2) `n` lớn: mọi DP khớp nhau; (3) **hoán vị** đầu vào không đổi đáp án; (4) **thêm từ trùng** không đổi đáp án; (5) **thêm từ** không làm đáp án giảm và tăng tối đa `1`... (chỉ chắc chắn không giảm); (6) **chuỗi gài sẵn**: dựng chuỗi dài `m` bằng cách chèn ngẫu nhiên từng chữ cái rồi trộn nhiễu ⇒ đáp án `≥ m`; (7) chuỗi khôi phục hợp lệ theo định nghĩa và có độ dài bằng đáp án; (8) `1 ≤ đáp án ≤ (max len) − (min len) + 1`.

In [23]:
def plant_chain(rng, m, alphabet="abcd"):
    w = rng.choice(alphabet)
    chain = [w]
    for _ in range(m - 1):
        i = rng.randint(0, len(w))
        w = w[:i] + rng.choice(alphabet) + w[i:]
        chain.append(w)
    return chain


rng = random.Random(20240148)
t0 = time.perf_counter()
for it in range(800):
    n = rng.randint(1, 11)
    alpha = rng.choice(["ab", "abc", "abcd"])
    words = ["".join(rng.choice(alpha) for _ in range(rng.randint(1, 5))) for _ in range(n)]
    check(words, brute=True)                                                        # (1)
print("(1) 800 danh sách nhỏ khớp oracle:", round(time.perf_counter() - t0, 1), "s")

t0 = time.perf_counter()
for it in range(300):
    m = rng.randint(1, 16)
    chain = plant_chain(rng, m)
    noise = ["".join(rng.choice("abcd") for _ in range(rng.randint(1, 16))) for _ in range(rng.randint(0, 150))]
    words = chain + noise
    rng.shuffle(words)
    ans = check(words)                                                              # (2)
    assert ans >= m                                                                 # (6)
    shuffled = words[:]
    rng.shuffle(shuffled)
    assert Solution().longestStrChain(shuffled) == ans                              # (3)
    assert Solution().longestStrChain(words + rng.sample(words, min(5, len(words)))) == ans   # (4)
    extra = "".join(rng.choice("abcd") for _ in range(rng.randint(1, 16)))
    assert Solution().longestStrChain(words + [extra]) >= ans                        # (5)
    ch = best_chain(words)
    assert len(ch) == ans and valid_chain(ch, set(words))                           # (7)
    lens = [len(w) for w in words]
    assert 1 <= ans <= max(lens) - min(lens) + 1                                    # (8)
print("(2)-(8) 300 danh sách gài chuỗi + nhiễu (tới 166 từ):", round(time.perf_counter() - t0, 1), "s")

(1) 800 danh sách nhỏ khớp oracle: 0.3 s


(2)-(8) 300 danh sách gài chuỗi + nhiễu (tới 166 từ): 3.2 s


### F. Giới hạn

`n = 1000`, độ dài `≤ 16`: (i) `1000` từ nhị phân độ dài `1..9` (nhiều quan hệ tiền bối, đáp án `9`); (ii) toàn từ `a^L` (`16` mức, mỗi mức lặp nhiều lần ⇒ đáp án `16`); (iii) `1000` từ ngẫu nhiên; (iv) chuỗi gài `16` bậc cộng `984` từ nhiễu; (v) mọi từ cùng độ dài `16` (đáp án `1`). Mọi cài đặt DP phải khớp và chạy nhanh.

In [24]:
t0 = time.perf_counter()
rg = random.Random(99)
cases = {
    "nhị phân ≤ 9": ([ "".join(t) for L in range(1, 10) for t in itertools.product("ab", repeat=L)][:1000], 9),
    "a^L lặp lại": (["a" * (1 + i % 16) for i in range(1000)], 16),
    "ngẫu nhiên": (["".join(rg.choice("abcdefgh") for _ in range(rg.randint(1, 16))) for _ in range(1000)], None),
    "chuỗi 16 + nhiễu": (plant_chain(rg, 16) + ["".join(rg.choice("xyz") for _ in range(rg.randint(1, 16))) for _ in range(984)], None),
    "cùng độ dài 16": (["".join(rg.choice("abc") for _ in range(16)) for _ in range(1000)], 1),
}
for name, (words, want) in cases.items():
    ans = check(words, want)
    print(f"  {name:18s}: n = {len(words)}, đáp án = {ans}")
assert cases["chuỗi 16 + nhiễu"][1] is None and Solution().longestStrChain(cases["chuỗi 16 + nhiễu"][0]) >= 16
print("n = 1000:", round(time.perf_counter() - t0, 1), "s")

  nhị phân ≤ 9      : n = 1000, đáp án = 9


  a^L lặp lại       : n = 1000, đáp án = 16


  ngẫu nhiên        : n = 1000, đáp án = 5


  chuỗi 16 + nhiễu  : n = 1000, đáp án = 16


  cùng độ dài 16    : n = 1000, đáp án = 1
n = 1000: 1.9 s


## 13. Các lỗi người mới hay gặp

### Lỗi 1: không sắp xếp theo độ dài

In [25]:
w = ["bca", "ba", "a"]
print("bug:", bug_no_sort(w), "| đúng:", Solution().longestStrChain(w))
assert bug_no_sort(w) == 1 and Solution().longestStrChain(w) == 3

bug: 1 | đúng: 3


**Vì sao sai:** DP chỉ đúng nếu khi xử lý `w` mọi tiền bối của nó đã có giá trị. Đầu vào đảo thứ tự (dài trước) làm `dp` của tiền bối chưa sẵn sàng. **Đúng:** `sorted(words, key=len)`.

### Lỗi 2: dãy con nhưng bỏ điều kiện hơn kém đúng `1` ký tự

In [26]:
w = ["a", "abc"]
print("bug:", bug_subsequence_only(w), "| đúng:", Solution().longestStrChain(w))
assert bug_subsequence_only(w) == 2 and Solution().longestStrChain(w) == 1

bug: 2 | đúng: 1


**Vì sao sai:** `"a"` là dãy con của `"abc"` nhưng cần chèn **hai** chữ cái, không phải một. Tiền bối đòi hỏi **thêm đúng một** chữ cái.

### Lỗi 3: chỉ xoá ký tự đầu hoặc cuối

In [27]:
w = ["a", "b", "ba", "bca", "bda", "bdca"]
print("bug:", bug_ends_only(w), "| đúng:", Solution().longestStrChain(w))
assert bug_ends_only(w) < Solution().longestStrChain(w) == 4

bug: 2 | đúng: 4


**Vì sao sai:** chèn được vào **bất kỳ** vị trí, kể cả giữa (`ba → bca`). Phải thử xoá mọi vị trí `0..len-1`.

### Lỗi 4: đếm số độ dài khác nhau thay vì chuỗi thật

In [28]:
w = ["abcd", "dbqca"]
print("bug:", bug_distinct_lengths(w), "| đúng:", Solution().longestStrChain(w))
assert bug_distinct_lengths(w) == 2 and Solution().longestStrChain(w) == 1

bug: 2 | đúng: 1


**Vì sao sai:** có từ ở các độ dài liên tiếp chưa chắc nối thành chuỗi: phải kiểm tra quan hệ tiền bối thực sự giữa các từ.

### Lỗi 5: quên tính chính từ đó (khởi tạo `0`)

In [29]:
w = ["a"]
print("bug:", bug_init_zero(w), "| đúng:", Solution().longestStrChain(w))
assert bug_init_zero(w) == 0 and Solution().longestStrChain(w) == 1

bug: 0 | đúng: 1


**Vì sao sai:** một từ đơn lẻ là chuỗi độ dài `1`; `dp[w]` phải bắt đầu từ `1`, không phải `0` (kết quả lệch `1` trên mọi test).

### Lỗi 6 (bonus): `KeyError` khi tiền bối không có trong từ điển

In [30]:
try:
    bug_keyerror(["a", "bb"])
except KeyError as e:
    print("KeyError:", e)
assert Solution().longestStrChain(["a", "bb"]) == 1

KeyError: ''


**Vì sao sai:** ứng viên `p = w[:i] + w[i+1:]` thường **không** là từ trong danh sách. Phải kiểm tra `p in dp` (hoặc dùng `dp.get(p, 0)`) trước khi đọc.

## 14. Mẹo & nhận dạng pattern

**Nhận dạng:** "chuỗi các từ, mỗi bước biến đổi nhỏ (thêm/xoá một ký tự), tìm chuỗi dài nhất" → **DP trên DAG theo độ dài**; ứng viên sinh bằng biến đổi ngược (xoá từng vị trí) thay vì thử mọi cặp.

**Mẹo:**

1. Sắp xếp theo độ dài (hoặc bucket theo độ dài) để đảm bảo thứ tự tô-pô.
2. Sinh `≤ L` tiền bối ứng viên bằng cắt chuỗi, tra bảng băm `O(1)` (trung bình).
3. `dp[w]` khởi tạo `1`; luôn kiểm tra `p in dp`.
4. Hai con trỏ cho kiểm tra "dãy con hơn kém đúng một" khi không muốn băm.
5. Oracle: DFS theo **định nghĩa chèn** trên danh sách nhỏ; kiểm tra ba định nghĩa tiền bối tương đương.
6. Khôi phục chuỗi bằng `parent[w]`.

**Họ bài cùng pattern:** Longest Increasing Subsequence (300), Word Ladder (127), Russian Doll Envelopes (354), Largest Divisible Subset (368), Number of Longest Increasing Subsequence (673), Delete Operation for Two Strings (583).

## 15. Biến thể

1. **Khôi phục một chuỗi dài nhất** (đã làm ở mục 8): `best_chain`.
2. **Đếm số chuỗi dài nhất** (chuỗi = dãy từ phân biệt): DP đếm song song.
3. **Tiền bối chỉ là tiền tố** (chỉ được thêm chữ ở **cuối** từ): bài đơn giản hơn, vẫn DP theo độ dài.

Cài đặt và đối chiếu vét cạn:

In [31]:
def count_longest_answer(words):
    ws = sorted(set(words), key=len)
    dp, cnt = {}, {}
    for w in ws:
        preds = {w[:i] + w[i + 1:] for i in range(len(w))} & dp.keys()      # tiền bối phân biệt
        if not preds:
            dp[w], cnt[w] = 1, 1
        else:
            top = max(dp[p] for p in preds)
            dp[w], cnt[w] = top + 1, sum(cnt[p] for p in preds if dp[p] == top)
    best = max(dp.values())
    return best, sum(cnt[w] for w in dp if dp[w] == best)


def brute_count_longest(words):
    ws = sorted(set(words))
    best, ways = 1, 0
    def dfs(cur, length):
        nonlocal best, ways
        if length > best:
            best, ways = length, 1
        elif length == best:
            ways += 1
        for nxt in ws:
            if pred_by_insert(cur, nxt):
                dfs(nxt, length + 1)
    best = 1
    for w in ws:
        dfs(w, 1)
    return best, ways


def prefix_chain(words):
    dp = {}
    for w in sorted(set(words), key=len):
        dp[w] = dp.get(w[:-1], 0) + 1
    return max(dp.values())


def brute_prefix_chain(words):
    ws = sorted(set(words))
    best = 1
    def dfs(cur, length):
        nonlocal best
        best = max(best, length)
        for nxt in ws:
            if len(nxt) == len(cur) + 1 and nxt[:-1] == cur:
                dfs(nxt, length + 1)
    for w in ws:
        dfs(w, 1)
    return best


rng = random.Random(15)
for _ in range(500):
    n = rng.randint(1, 10)
    words = ["".join(rng.choice("ab") for _ in range(rng.randint(1, 5))) for _ in range(n)]
    best, ways = brute_count_longest(words)
    assert best == chain_brute(words)
    assert count_longest_answer(words) == (best, ways), words
    assert prefix_chain(words) == brute_prefix_chain(words), words
print("đếm số chuỗi dài nhất và biến thể tiền tố: khớp vét cạn")

đếm số chuỗi dài nhất và biến thể tiền tố: khớp vét cạn


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. DFS vét cạn | thử mọi chuỗi | mũ | `O(n)` | oracle |
| 2. Hai con trỏ | so mọi cặp `j < i` | `O(n² L)` | `O(n)` | đủ cho `n ≤ 1000` |
| 3. Băm + xoá ký tự | `p = w` xoá 1 ký tự | `O(n L²)` | `O(n L)` | **đáp án chính** |
| 4. Đệ quy có nhớ | `go(w)` tiền bối trong tập | `O(n L²)` | `O(n L)` | không cần sắp xếp |
| 5. Đẩy: chèn chữ cái | từ `w` sinh hậu duệ | `O(26 n L²)` | `O(n L)` | cách nhìn hậu duệ |

Mọi cách DP khớp oracle trên `20 478` tập từ nhỏ và `1 100` danh sách ngẫu nhiên / gài chuỗi, khớp nhau trên `n = 1000`.

## 17. Bài tập vận dụng

Các cell TODO **chưa được gọi**: tự điền rồi bỏ comment dòng test cuối cell.

### Bài 1 (đoán output)

Đoán trước rồi chạy:

In [32]:
print(Solution().longestStrChain(["a", "ab", "abc", "abcd"]), Solution().longestStrChain(["abc", "ab", "a"]), Solution().longestStrChain(["ab", "ba"]), Solution().longestStrChain(["a", "b", "c", "ab", "ac", "abc"]))

4 3 1 3


### Bài 2 (tính tay)

`words = ["a","ab","ac","bd","abc","abd","abdd"]`. Sắp theo độ dài và lập bảng `dp` bằng tay (ghi các tiền bối ứng viên có trong từ điển). Kiểm tra ở mục 19.

### Bài 3 (điền TODO)

Hoàn thiện DP bảng băm:

In [33]:
def chain_todo(words):
    dp = {}
    best = 1
    for w in sorted(words, key=len):
        cur = 1
        for i in range(len(w)):
            pass        # TODO 1: p = w bỏ ký tự thứ i; nếu p in dp thì cur = max(cur, dp[p] + 1)
        dp[w] = cur
        best = max(best, cur)
    return best


# Khi làm xong, bỏ comment để kiểm tra:
# assert all(chain_todo(list(w)) == v for w, v in examples)

### Bài 4 (tự viết lại)

Không nhìn lời giải, viết `class MySolution` bằng **đệ quy có nhớ** hoặc **hai con trỏ**:

In [34]:
class MySolution:
    def longestStrChain(self, words):
        pass            # TODO


# Khi làm xong:
# assert all(MySolution().longestStrChain(list(w)) == v for w, v in examples)

### Bài 5 (mini challenge): đếm số chuỗi dài nhất

Viết `count_longest_chains(words)` trả về cặp `(độ dài dài nhất, số chuỗi đạt độ dài đó)` (chuỗi là dãy các từ **phân biệt**). Gợi ý: đếm song song với `dp`, gộp theo các tiền bối **phân biệt** (xoá hai vị trí khác nhau có thể cho cùng một chuỗi, ví dụ `aab` xoá vị trí `0` hay `1` đều ra `ab`).

In [35]:
def count_longest_chains(words):
    pass            # TODO


# Khi làm xong:
# assert count_longest_chains(["a", "b", "ab", "ba", "aba"]) == (3, 4)
# assert count_longest_chains(["a", "aa", "aaa"]) == (3, 1)

## 18. Gợi ý 3 mức

**Bài 1**

- Mức 1: sắp theo độ dài rồi DP; xoá từng ký tự để tìm tiền bối.
- Mức 2: `["a","ab","abc","abcd"]` là một chuỗi hoàn chỉnh; `["ab","ba"]` cùng độ dài nên không nối nhau.
- Mức 3: `4 3 1 3`.

**Bài 2**

- Mức 1: độ dài `1`: `a` (1); độ dài `2`: `ab`, `ac`, `bd`.
- Mức 2: `ab` có tiền bối `a` (1) ⇒ 2; `ac` ⇒ 2; `bd`: ứng viên `d`, `b` không có trong danh sách ⇒ 1. `abc`: ứng viên `bc` (không có), `ac`(2), `ab`(2) ⇒ 3; `abd`: ứng viên `bd`(1), `ad` (không có), `ab`(2) ⇒ 3; `abdd`: ứng viên `bdd`, `add` (không có), `abd`(3) (xoá vị trí `2` hoặc `3` đều ra `abd`) ⇒ 4.
- Mức 3: đáp án `4` (`a → ab → abd → abdd`).

**Bài 3**

- Mức 1: `p = w[:i] + w[i + 1:]`.
- Mức 2: `if p in dp:`.
- Mức 3: `cur = max(cur, dp[p] + 1)`.

**Bài 4**

- Mức 1: `S = set(words)`; `go(w)` trả về chuỗi dài nhất kết thúc tại `w`.
- Mức 2: `go(w) = 1 + max(go(p))` trên các `p ∈ S` thu được khi xoá một ký tự (mặc định `1`).
- Mức 3: dùng `lru_cache`; đáp án `max(go(w) for w in S)`.

**Bài 5**

- Mức 1: giữ `cnt[w]` = số chuỗi dài nhất kết thúc tại `w`.
- Mức 2: lấy tập tiền bối **phân biệt** `{w[:i] + w[i+1:]} ∩ dp`; `top` = max `dp[p]`.
- Mức 3: `dp[w] = top + 1`, `cnt[w] = Σ cnt[p]` cho `p` có `dp[p] = top` (không có tiền bối thì `dp = cnt = 1`); đáp án gộp các `w` có `dp` lớn nhất.

## 19. Đáp án tham khảo

Mỗi đáp án đều có `assert`.

### Bài 1

In [36]:
vals = (Solution().longestStrChain(["a", "ab", "abc", "abcd"]), Solution().longestStrChain(["abc", "ab", "a"]), Solution().longestStrChain(["ab", "ba"]), Solution().longestStrChain(["a", "b", "c", "ab", "ac", "abc"]))
print(vals)
assert vals == (4, 3, 1, 3)
assert vals == (chain_brute(["a", "ab", "abc", "abcd"]), chain_brute(["abc", "ab", "a"]), chain_brute(["ab", "ba"]), chain_brute(["a", "b", "c", "ab", "ac", "abc"]))
print("Bài 1 OK")

(4, 3, 1, 3)
Bài 1 OK


### Bài 2

In [37]:
w = ["a", "ab", "ac", "bd", "abc", "abd", "abdd"]
assert trace_hash(w) == 4 == chain_brute(w)
ch = best_chain(w)
print(ch)
assert len(ch) == 4 and valid_chain(ch, w)
print("Bài 2 OK")

words = ['a', 'ab', 'ac', 'bd', 'abc', 'abd', 'abdd']
  'a'      : ứng viên [''] -> có trong dp (không) -> dp = 1
  'ab'     : ứng viên ['b', 'a'] -> có trong dp {'a': 1} -> dp = 2
  'ac'     : ứng viên ['c', 'a'] -> có trong dp {'a': 1} -> dp = 2
  'bd'     : ứng viên ['d', 'b'] -> có trong dp (không) -> dp = 1
  'abc'    : ứng viên ['bc', 'ac', 'ab'] -> có trong dp {'ac': 2, 'ab': 2} -> dp = 3
  'abd'    : ứng viên ['bd', 'ad', 'ab'] -> có trong dp {'bd': 1, 'ab': 2} -> dp = 3
  'abdd'   : ứng viên ['bdd', 'add', 'abd', 'abd'] -> có trong dp {'abd': 3} -> dp = 4
  đáp án = 4
['a', 'ab', 'abd', 'abdd']
Bài 2 OK


### Bài 3

In [38]:
def chain_answer(words):
    dp = {}
    best = 1
    for w in sorted(words, key=len):
        cur = 1
        for i in range(len(w)):
            p = w[:i] + w[i + 1:]
            if p in dp:
                cur = max(cur, dp[p] + 1)
        dp[w] = cur
        best = max(best, cur)
    return best


assert all(chain_answer(list(w)) == v for w, v in examples)
for words in pool[:1500]:
    assert chain_answer(list(words)) == chain_brute(words)
print("Bài 3 OK")

Bài 3 OK


### Bài 4

In [39]:
class MySolutionAnswer:
    def longestStrChain(self, words):
        S = set(words)

        @lru_cache(maxsize=None)
        def go(w):
            best = 1
            for i in range(len(w)):
                p = w[:i] + w[i + 1:]
                if p in S:
                    best = max(best, go(p) + 1)
            return best

        return max(go(w) for w in S)


assert all(MySolutionAnswer().longestStrChain(list(w)) == v for w, v in examples)
for words in pool[:1500]:
    assert MySolutionAnswer().longestStrChain(list(words)) == chain_brute(words)
print("Bài 4 OK")

Bài 4 OK


### Bài 5

In [40]:
def count_longest_chains(words):
    return count_longest_answer(words)


assert count_longest_chains(["a", "b", "ab", "ba", "aba"]) == (3, 4)
assert count_longest_chains(["a", "aa", "aaa"]) == (3, 1)
assert count_longest_chains(["a", "aab", "ab"]) == (3, 1)          # aab xoá vị trí 0 hoặc 1 đều ra "ab": chỉ tính một chuỗi
for words in pool[:1500]:
    assert count_longest_chains(words) == brute_count_longest(words)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (chuỗi từ bước nhảy "thêm/xoá một ký tự", tìm chuỗi dài nhất):**

```python
class Solution:
    def longestStrChain(self, words: List[str]) -> int:
        dp = {}
        best = 1
        for w in sorted(words, key=len):
            cur = 1
            for i in range(len(w)):
                p = w[:i] + w[i + 1:]
                if p in dp:
                    cur = max(cur, dp[p] + 1)
            dp[w] = cur
            best = max(best, cur)
        return best
```

**Checklist trước khi submit:**

- [ ] Đã sắp xếp theo độ dài tăng dần trước khi DP?
- [ ] Tiền bối sinh bằng xoá **mọi** vị trí `0..len-1` (không chỉ đầu/cuối)?
- [ ] Kiểm tra `p in dp` trước khi đọc `dp[p]` (không `KeyError`)?
- [ ] `dp[w]` khởi tạo `1`, đáp án là `max` của mọi `dp[w]`?
- [ ] Từ trùng, mọi từ cùng độ dài, từ đơn lẻ: đáp án `1`?
- [ ] Đúng tên `class Solution`, `longestStrChain(self, words)`, không in debug.